# P2 — Preparación supervisada y verificación del split

## Propósito
Construir una partición reproducible, balanceada y libre de fugas de información.

Se utiliza:

- 70 % TRAIN;
- 15 % VALIDATION;
- 15 % TEST interno.

In [ ]:

from pathlib import Path
import sys
import yaml

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró config.yaml. "
        "Ejecute Jupyter desde la carpeta del proyecto o notebooks/."
    )

ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(
    (ROOT / "config.yaml").read_text(encoding="utf-8")
)

print("Proyecto:", ROOT)


## 1. Cargar partición

In [ ]:
import pandas as pd

split_df = pd.read_csv(
    ROOT / cfg["data"]["supervised_split"]
)

display(split_df.head(10))

## 2. Balance por clase y split

In [ ]:
balance = (
    split_df.groupby(["split","label"])
    .size()
    .rename("n")
    .reset_index()
)

display(balance)

## 3. Verificar rutas relativas

In [ ]:
from src.data import validate_paths

missing = validate_paths(
    split_df,
    ROOT
)

print("Rutas faltantes:", len(missing))

if missing:
    display(
        pd.DataFrame(
            {"missing_path": missing[:20]}
        )
    )
else:
    print("Todas las imágenes fueron localizadas correctamente.")

## 4. Control de fuga de información

In [ ]:
split_count = (
    split_df.groupby("path")["split"]
    .nunique()
)

leaks = split_count[
    split_count > 1
]

print("Imágenes presentes en más de un split:", len(leaks))

assert len(leaks) == 0

## 5. Proporciones reales

In [ ]:
split_summary = (
    split_df.groupby("split")
    .size()
    .rename("n")
    .reset_index()
)

split_summary["percent"] = (
    100 * split_summary["n"] /
    split_summary["n"].sum()
).round(2)

display(split_summary)

## Exportación del protocolo de partición


In [ ]:
OUT = ROOT / "results/audit"
OUT.mkdir(parents=True, exist_ok=True)

balance.to_csv(OUT / "split_class_balance.csv", index=False)
split_summary.to_csv(OUT / "split_summary.csv", index=False)

path_check = split_df[["path","label","split"]].copy()
path_check["exists"] = path_check["path"].map(lambda p: (ROOT/str(p)).exists())
path_check.to_csv(OUT / "split_path_validation.csv", index=False)

leak_df = pd.DataFrame({"path": leaks.index.astype(str), "n_splits": leaks.values}) if len(leaks) else pd.DataFrame(columns=["path","n_splits"])
leak_df.to_csv(OUT / "split_leakage_check.csv", index=False)

assert path_check["exists"].all(), "Existen rutas faltantes en el split."
assert len(leaks)==0, "Se detectó fuga de imágenes entre particiones."
print('P2 exportó balance, resumen, validación de rutas y control de fuga.')


In [ ]:
from PIL import Image, ImageOps
import numpy as np

def _dhash256(path, hash_size=16):
    im = ImageOps.exif_transpose(Image.open(path)).convert("L").resize((hash_size+1, hash_size))
    a = np.asarray(im)
    return (a[:,1:] > a[:,:-1]).ravel()

_H = np.array([_dhash256(ROOT / p) for p in split_df["path"].astype(str)], dtype=bool)
_rows = []
for i, r in split_df.reset_index(drop=True).iterrows():
    other = np.where(split_df.reset_index(drop=True)["split"].to_numpy() != r["split"])[0]
    d = np.count_nonzero(_H[other] != _H[i], axis=1)
    j = int(other[int(np.argmin(d))])
    rr = split_df.reset_index(drop=True).iloc[j]
    _rows.append({
        "path": r["path"], "split": r["split"], "label": int(r["label"]),
        "nearest_cross_split_path": rr["path"], "nearest_cross_split_split": rr["split"],
        "nearest_cross_split_label": int(rr["label"]), "dhash256_hamming": int(d.min())
    })
perceptual = pd.DataFrame(_rows)
perceptual.to_csv(OUT / "cross_split_perceptual_similarity.csv", index=False)
print("dHash cross-split mínimo:", int(perceptual["dhash256_hamming"].min()), "/ 256")
display(perceptual.nsmallest(10, "dhash256_hamming"))


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

_features = ["brightness_mean", "brightness_std"]
_tr = split_df[split_df["split"] == "train"]
_te = split_df[split_df["split"] == "test_internal"]
_photo = make_pipeline(StandardScaler(), LogisticRegression(random_state=int(cfg["project"]["seed"]), max_iter=1000))
_photo.fit(_tr[_features], _tr["label"])
_pp = _photo.predict_proba(_te[_features])[:,1]
_py = (_pp >= 0.5).astype(int)
_cv = cross_val_score(
    make_pipeline(StandardScaler(), LogisticRegression(random_state=int(cfg["project"]["seed"]), max_iter=1000)),
    split_df[_features], split_df["label"],
    cv=StratifiedKFold(5, shuffle=True, random_state=int(cfg["project"]["seed"])), scoring="accuracy")
_photo_summary = pd.DataFrame([{
    "diagnostic":"brightness_mean + brightness_std logistic regression",
    "train_n":len(_tr), "test_n":len(_te),
    "test_accuracy":accuracy_score(_te["label"],_py),
    "test_f1":f1_score(_te["label"],_py),
    "test_roc_auc":roc_auc_score(_te["label"],_pp),
    "cv5_accuracy_mean":float(_cv.mean()), "cv5_accuracy_std":float(_cv.std()),
    "interpretation":"Potential photometric shortcut; diagnostic only, not a competing vision model."
}])
_photo_summary.to_csv(OUT/"photometric_shortcut_baseline.csv",index=False)
pd.DataFrame({"path":_te["path"].values,"label":_te["label"].values,
              "brightness_mean":_te["brightness_mean"].values,"brightness_std":_te["brightness_std"].values,
              "prob_anomaly_photometric":_pp,"pred":_py}).to_csv(OUT/"photometric_shortcut_test_predictions.csv",index=False)
display(_photo_summary)
